# Pipeline

https://github.com/datamindedbe/blog-tpcds-dbt-duckdb/tree/main

```
uv sync
```

In [ ]:
# # run this to generate index for values in the hierarchy yaml files

# import duckdb
# from src.hierarchy_duckdb import build_tree_with_stats
# from pathlib import Path
# proj_path = Path().resolve()
# data_path = proj_path / 'data'
# duckdb_conn = duckdb.connect(database=str(proj_path / 'tpcds/tpcds.db'))
# index_path = data_path / 'index'
# for yaml_path in (data_path / 'hierarchy').glob('*.yaml'):
#     tree = build_tree_with_stats(yaml_path, index_path, duckdb_conn)
#     with (data_path / 'hierarchy' / f"{yaml_path.stem}.json").open('w') as f:
#         f.write(tree.to_json())

In [ ]:
# # run this only once to generate the TPC-DS data
import duckdb

con = duckdb.connect(database='./tpcds/tpcds.db')
# con = duckdb.connect(
#     database='./cube-project/data/tpcds.db',
#     read_only=True,
# )
# con.execute('INSTALL tpcds;')
# con.execute('LOAD tpcds;')
# con.execute("CALL dsdgen(sf = 1);")  # run only once generate data with scale factor 1 (1GB)

In [3]:
df = con.execute("""SELECT ca_zip, Sum(cs_sales_price) 
FROM   catalog_sales, 
       customer, 
       customer_address, 
       date_dim 
WHERE  cs_bill_customer_sk = c_customer_sk 
       AND c_current_addr_sk = ca_address_sk 
       AND ( Substr(ca_zip, 1, 5) IN ( '85669', '86197', '88274', '83405', 
                                       '86475', '85392', '85460', '80348', 
                                       '81792' ) 
              OR ca_state IN ( 'CA', 'WA', 'GA' ) 
              OR cs_sales_price > 500 ) 
       AND cs_sold_date_sk = d_date_sk 
       AND d_qoy = 1 
       AND d_year = 1998 
GROUP  BY ca_zip 
ORDER  BY ca_zip
LIMIT 100; 
""").fetch_df()
df.head()

,ca_zip,sum(cs_sales_price)
0,30069,1603.94
1,30150,1030.54
2,30162,408.90
3,30169,855.10
4,30399,224.72


Ghita

# Schema Graph

In [1]:
import sys
from pathlib import Path
proj_path = Path().resolve()
sys.path.append(str(proj_path / 'src'))

from src.schema_processor import display_graph
db_type = 'tutorial'  # 'tutorial' or 'tpcds'
data_path = proj_path / 'data' / db_type

In [2]:
display_graph(data_path)

Error initializing SchemaExplorer: Could not locate schema graph JSONs relative to /home/jsjang/code/Agent4OLAP/data/tutorial. Checked: /home/jsjang/code/Agent4OLAP/data/tutorial, /home/jsjang/code/Agent4OLAP/data
 Make sure to run create_schema_graphs() first: ```python src/schema_processor.py --create_graphs``` 


FileNotFoundError: Could not locate schema graph JSONs relative to /home/jsjang/code/Agent4OLAP/data/tutorial. Checked: /home/jsjang/code/Agent4OLAP/data/tutorial, /home/jsjang/code/Agent4OLAP/data

In [ ]:
from src.schema_processor import SchemaExplorer

explorer = SchemaExplorer(data_path)
print(explorer.get_facts())
print(explorer.get_schema('star', 'catalog_sales'))
print()
# TODO: need from/target searching
attr_results = explorer.search_attribute('star', 'catalog_sales', 'ca_state')  # d_fy_year, s_store_id
attr_results

In [ ]:
import random
from src.unique_index import UniqueIndex
path = './data/index/date_dim__d_fy_year'
idx = UniqueIndex(path, fast=False)

x = random.sample(list(iter(idx)), k=1)[0]
print("Search for:", x)
o = explorer.search_value('star', 'store_sales', 'd_fy_year', x)
print("Found:", o[0]['found'])
o

----

## About the TPC-DS queries

In [ ]:
from pathlib import Path
from collections import defaultdict
from src.schema_processor import SchemaExplorer
queries_path = Path('./queries/tpcds')

fact2queries = defaultdict(list)
queries2fact = defaultdict(set)
for qpath in queries_path.glob('*.sql'):
    with qpath.open() as f:
        sql = f.read()
    for fact in SchemaExplorer.tpcds_facts:
        if fact in sql.lower():
            queries2fact[qpath.stem].add(fact)

for query, facts in queries2fact.items():
    # set the number of the facts as key, more or equal to 4 make them one group
    if len(facts) >= 4:
        fact2queries['4+'].append(query)
    else:
        fact2queries[str(len(facts))].append(query)

In [ ]:
for k, v in sorted(fact2queries.items(), key=lambda x: (int(x[0]) if x[0].isdigit() else 99)):
    print(f"{k}: {len(v)}")

In [ ]:
queries2fact['query15']

In [ ]:
facts_queries_by_numbers: dict[str, dict[str, list[str]]] = defaultdict(dict)
for fact in SchemaExplorer.tpcds_facts:
    for number, queries in fact2queries.items():
        if facts_queries_by_numbers[fact].get(number) is None:
            facts_queries_by_numbers[fact][number] = []
        facts_queries_by_numbers[fact][number].extend(queries)

In [ ]:
sorted(facts_queries_by_numbers['store_sales']['1'])[:5]

In [ ]:
# plot the distribution of number of fact tables per query
# make the number in the center of the bars
# if number is 4 or more, put it in 4+
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

fact_counts = [len(v) for v in queries2fact.values()]
fact_counts = [4 if x >= 4 else x for x in fact_counts]
fig, ax = plt.subplots()

sns.histplot(fact_counts, discrete=True, shrink=0.9, ax=ax)
ax.set_xlabel('Number of Fact Tables in a Query')
ax.set_ylabel('Number of Queries')
ax.set_title('Distribution of Number of Fact Tables per Query')
plt.xticks(ticks=[1, 2, 3, 4], labels=['1', '2', '3', '4+'])
plt.grid(axis='y')
plt.show()

---

In [1]:
from src.schema_processor import SchemaExplorer
from pathlib import Path
data_path = Path("./data/tutorial")
explorer = SchemaExplorer(data_path)

In [2]:
explorer.get_facts()

{'fact_sales'}

In [3]:
explorer.get_schema('star', 'fact_sales')

[{'id': 'fact_sales', 'type': 'fact'},
 {'id': 'dim_date', 'type': 'dimension'},
 {'id': 'dim_product', 'type': 'dimension'},
 {'id': 'dim_store', 'type': 'dimension'}]

In [4]:
explorer.get_attributes('star', 'fact_sales', 'dim_date')

['date_key', 'date', 'week', 'month', 'quarter', 'year']

In [8]:
explorer.search_attribute('star', 'fact_sales', 'week')

[{'dimension': 'dim_date',
  'attribute': 'week',
  'path': [{'type': 'dimension', 'name': 'dim_date', 'label': 'dim_date'},
   {'type': 'level', 'name': 'date', 'label': 'Date'},
   {'type': 'attribute', 'name': 'week', 'label': 'Week'}],
  'stats': {'count': 365,
   'null_count': 0,
   'distinct_count': 52,
   'min': 1,
   'max': 52,
   'range': [1, 52],
   'dtype': 'integer',
   'unique_values': {'type': 'bplustree',
    'values': '/home/jsjang/code/Agent4OLAP/data/tutorial/index/Dim_Date__week'}}}]

In [10]:
explorer.search_value('star', 'fact_sales', 'week', 3)

[{'match': {'dimension': 'dim_date',
   'attribute': 'week',
   'path': [{'type': 'dimension', 'name': 'dim_date', 'label': 'dim_date'},
    {'type': 'level', 'name': 'date', 'label': 'Date'},
    {'type': 'attribute', 'name': 'week', 'label': 'Week'}],
   'stats': {'count': 365,
    'null_count': 0,
    'distinct_count': 52,
    'min': 1,
    'max': 52,
    'range': [1, 52],
    'dtype': 'integer',
    'unique_values': {'type': 'bplustree',
     'values': '/home/jsjang/code/Agent4OLAP/data/tutorial/index/Dim_Date__week'}}},
  'value': 3,
  'found': True}]